# SmolVLA Model/Data Diagnosis: A -> E

Notebook này chạy đủ 5 section trước khi làm VLA Harness runtime.

**Lưu ý commit HF:** mỗi lần upload model có 3 commit. Phải dùng commit cuối mỗi cụm để có đủ weights/config + preprocessor + postprocessor.

- Old final: `b6f2aafdbdd793046747fad8207459402c33c4b0`
- New final: `f7029d03d69e149cb4b7cea8747d7158d35a8fd0`

## 0. Setup

Nếu không chạy ở `/home/trietlm/lerobot`, set env `LEROBOT_ROOT` hoặc sửa `cfg.repo_root`. Nếu dataset không ở path mặc định, set env `LEROBOT_DATASET_ROOT`.

In [ ]:
from pathlib import Path
import sys
from IPython.display import Markdown, display

REPO_ROOT = Path('/home/trietlm/lerobot')
if not REPO_ROOT.exists():
    REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'xai'))

from smolvla_model_data_diagnosis import (
    DiagnosisConfig, OLD_FINAL_REV, NEW_FINAL_REV, choose_probe_rows,
    run_replay, run_state_scan, run_dataset_audit, run_config_audit, write_eval_report,
)

cfg = DiagnosisConfig(repo_root=REPO_ROOT)
print('repo_root:', cfg.repo_root)
print('output_dir:', cfg.output_dir)
print('device:', cfg.device)
print('run_dirs:', [p.name for p in cfg.run_dirs])
print('dataset_root:', cfg.dataset_root, 'exists=', cfg.dataset_root.exists())
print('safe_pose:', cfg.safe_pose_path, 'exists=', cfg.safe_pose_path.exists())
print('policy specs:', cfg.policy_specs)
for run_dir in cfg.run_dirs:
    probes = choose_probe_rows(run_dir)
    print(run_dir.name, 'n_probes=', len(probes), 'first_timesteps=', [p['timestep'] for p in probes[:20]])

## A. Old vs New Replay

Replay old/new trên cùng recorded obs. Section này xuất `A_replay_predictions.csv`, `A_replay_summary.csv`, `A_server_compare.csv`, và plots vào `plots_A_replay/`.

In [ ]:
replay_df, replay_summary, run_policy_agg, server_compare = run_replay(cfg)
print('replay rows:', len(replay_df), 'errors:', int(replay_df['error'].notna().sum()))
display(run_policy_agg)
if not server_compare.empty:
    display(
        server_compare.groupby(['run', 'server_rtc_enabled', 'policy', 'image_variant'], dropna=False)
        .agg(
            n=('obs_timestep', 'count'),
            abs_delta_end_mean=('delta_end_server_minus_offline', lambda x: float(abs(x).mean())),
            delta_end_mean=('delta_end_server_minus_offline', 'mean'),
            server_safe_pull_count=('server_safe_pull', 'sum'),
            offline_safe_pull_count=('offline_safe_pull', 'sum'),
        )
        .reset_index()
    )
    display(server_compare.sort_values('delta_end_server_minus_offline', key=lambda s: s.abs(), ascending=False).head(40))

## B. State Scan

Giữ ảnh cố định, quét state từ `start_pose -> safe_pose` để tìm vùng state khiến model kéo safe.

In [ ]:
state_scan_df, state_scan_summary, danger = run_state_scan(cfg)
print('state scan rows:', len(state_scan_df), 'errors:', int(state_scan_df['error'].notna().sum()))
display(danger)
display(state_scan_summary.head())

## C. Dataset Bucket Audit

Đọc train parquet, project state/action theo trục `episode_start -> safe_pose`, rồi tìm bucket/episode có action kéo về safe/rest. Nếu dataset root thiếu, sửa `cfg.dataset_root` hoặc set `cfg.download_dataset_if_missing=True`.

In [ ]:
train_df, dataset_audit, bucket_summary, episode_suspects = run_dataset_audit(cfg)
print('train rows:', len(train_df), 'audit rows:', len(dataset_audit))
if not bucket_summary.empty:
    display(bucket_summary)
if not episode_suspects.empty:
    display(episode_suspects.head(40))

## D. Config / Processor Audit

So old/new final revisions về files, config, preprocessor/postprocessor. Đây là nơi kiểm tra có bị load nhầm commit giữa cụm 3 commit không.

In [ ]:
snapshot_df, file_manifest, config_flat, config_diff, file_diff = run_config_audit(cfg)
display(snapshot_df)
print('tracked files:', len(file_manifest), 'changed/missing files:', len(file_diff), 'changed config keys:', len(config_diff))
display(file_diff.head(100))
display(config_diff.head(200))

## E. Offline Eval Report

Gom các section thành report markdown + policy score.

In [ ]:
report_text = write_eval_report(
    cfg,
    run_policy_agg=run_policy_agg,
    danger=danger,
    suspects=episode_suspects,
    config_diff=config_diff,
    file_diff=file_diff,
)
print('wrote:', cfg.output_dir / 'E_offline_eval_report.md')
display(Markdown(report_text))